# FlowCast — M4 Exploratory Data Analysis

Northline Corridor | M4 EDA & Reporting

This notebook analyzes the processed M3 dataset without modifying it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data/processed/flowcast_m3_features.csv").exists():
    ROOT = ROOT.parent

df = pd.read_csv(ROOT / "data/processed/flowcast_m3_features.csv")
df["timestamp"] = pd.to_datetime(df["date"] + " " + df["time"])
sns.set_theme(style="whitegrid")

print("Shape:", df.shape)
print("Road segments:", df.road_id.nunique())
print("Date range:", df.timestamp.min(), "to", df.timestamp.max())


## 1. Dataset overview

In [ ]:
display(df.head())
display(df.describe(include="all").T)
print("Duplicate road-time keys:", df.duplicated(["road_id", "date", "time"]).sum())


## 2. Data quality and missing values

In [ ]:
nulls = df.isna().sum()
display(nulls[nulls > 0].to_frame("Missing values"))
print("Total null cells:", int(nulls.sum()))
print("These lag/rolling nulls are historical warm-up periods.")


## 3. Traffic volume distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df.traffic_volume, bins=50, kde=True)
plt.title("Traffic Volume Distribution")
plt.show()
print(df.traffic_volume.describe())


## 4. Congestion distribution

In [ ]:
order = ["Free-flow", "Moderate", "Heavy", "Severe"]
display(df.congestion_level.value_counts().reindex(order))
sns.countplot(data=df, x="congestion_level", order=order)
plt.title("Congestion Level Distribution")
plt.show()


## 5. Average speed distribution

In [ ]:
sns.histplot(df.avg_speed, bins=50, kde=True)
plt.title("Average Speed Distribution")
plt.show()


## 6. Traffic patterns by hour and weekday

In [ ]:
df.groupby("hour").traffic_volume.mean().plot(figsize=(10,5), marker="o")
plt.title("Average Traffic Volume by Hour")
plt.show()

daily = df.groupby("day_of_week").traffic_volume.mean()
daily.index = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
daily.plot(kind="bar", figsize=(10,5), title="Average Traffic by Weekday")
plt.show()


## 7. Road segment comparison

In [ ]:
road = df.groupby("road_name").traffic_volume.mean().sort_values()
road.plot(kind="barh", figsize=(10,7), title="Average Traffic by Road Segment")
plt.show()


## 8. Weather versus traffic

In [ ]:
weather = df.groupby("weather_condition").traffic_volume.agg(["mean","median","count"])
display(weather)
sns.barplot(data=df, x="weather_condition", y="traffic_volume", estimator="mean", errorbar=None)
plt.title("Average Traffic by Weather Condition")
plt.show()


## 9. Correlation analysis

In [ ]:
numeric = df.select_dtypes(include=np.number)
corr = numeric.corr()
plt.figure(figsize=(18,14))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlation Heatmap")
plt.show()
display(corr["traffic_volume"].sort_values(ascending=False).to_frame("Correlation"))


### Interpretation and leakage warning

`vehicle_count` has a correlation of approximately 0.9993 with `traffic_volume`. The Data Dictionary says it equals traffic volume in this release. Exclude it from traffic-volume model predictors. Correlation is association, not proof of causation.

## 10. Congestion by hour

In [ ]:
sns.countplot(data=df, x="hour", hue="congestion_level", hue_order=order)
plt.title("Congestion Levels by Hour")
plt.show()


## 11. Traffic volume versus average speed

In [ ]:
sample = df.sample(n=min(12000, len(df)), random_state=42)
sns.scatterplot(data=sample, x="traffic_volume", y="avg_speed", hue="congestion_level", alpha=.45, s=18)
plt.title("Traffic Volume versus Average Speed")
plt.show()
print("Pearson correlation:", df.traffic_volume.corr(df.avg_speed))


## 12. Modelling considerations and conclusions

- Use chronological train/test splitting.
- Preserve road-segment boundaries when constructing lag features.
- Do not randomly split time-series observations.
- Handle lag and rolling warm-up nulls without introducing leakage.
- Exclude `vehicle_count` from traffic-volume predictors.
- Investigate highly correlated predictors before model fitting.
- Treat correlation as association, not proof of causation.

**M4 output:** distributions, correlations, visualizations and data-quality observations.